In [1]:

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from lab2_utils import (
    get_final_data,
    get_split,
    balanced_weights,
    cv_macro_f1,
    CLASS_NAMES
)

# 1. Load the final prepared dataset
X_final, y_final = get_final_data()

# 2. Stratified 80/20 train-test split
X_train_final, X_test_final, y_train_final, y_test_final = get_split(
    X_final, y_final
)

# 3. Repeated stratified 5-fold cross-validation
rcv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

# 4. Create the initial XGBoost model
model = XGBClassifier(random_state=42)

# 5. Cross-validation with balanced sample weights
cv_scores_final = cv_macro_f1(
    model,
    X_train_final,
    y_train_final,
    cv=rcv,
    weights=True
)

print("CROSS-VALIDATION RESULTS")
print("Individual fold scores:", np.round(cv_scores_final, 4))
print(f"Mean CV Macro-F1: {cv_scores_final.mean():.4f}")
print(f"Std CV Macro-F1: {cv_scores_final.std():.4f}")

# 6. Train the model on the complete training set
final_model = clone(model)

final_model.fit(
    X_train_final,
    y_train_final,
    sample_weight=balanced_weights(y_train_final)
)

# 7. Generate predictions on the untouched test set
y_test_pred = final_model.predict(X_test_final)

# 8. Calculate test metrics
accuracy = accuracy_score(y_test_final, y_test_pred)

macro_precision = precision_score(
    y_test_final, y_test_pred, average="macro", zero_division=0
)

macro_recall = recall_score(
    y_test_final, y_test_pred, average="macro", zero_division=0
)

test_macro_f1 = f1_score(
    y_test_final, y_test_pred, average="macro", zero_division=0
)

print("\nTEST SET RESULTS")
print(f"Accuracy:         {accuracy:.4f}")
print(f"Macro Precision:  {macro_precision:.4f}")
print(f"Macro Recall:     {macro_recall:.4f}")
print(f"Test Macro-F1:    {test_macro_f1:.4f}")

# 9. Per-class precision, recall and F1
print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test_final,
    y_test_pred,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

# 10. Model information
print("\nMODEL INFORMATION")
print("Training samples:", len(X_train_final))
print("Test samples:", len(X_test_final))
print("Number of features:", X_final.shape[1])
print("Initial XGBoost model trained successfully.")


CROSS-VALIDATION RESULTS
Individual fold scores: [0.7773 0.7341 0.7294 0.7575 0.7327 0.7439 0.751  0.7303 0.7449 0.7644
 0.7549 0.7694 0.7505 0.7467 0.7229]
Mean CV Macro-F1: 0.7473
Std CV Macro-F1: 0.0152

TEST SET RESULTS
Accuracy:         0.7628
Macro Precision:  0.7457
Macro Recall:     0.7471
Test Macro-F1:    0.7452

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       <100k     0.7234    0.7083    0.7158        48
    100–130k     0.6542    0.6863    0.6699       102
    130–180k     0.7804    0.8068    0.7933       207
    180–250k     0.8305    0.7153    0.7686       137
    250–400k     0.7931    0.8734    0.8313        79
       ≥400k     0.6923    0.6923    0.6923        13

    accuracy                         0.7628       586
   macro avg     0.7457    0.7471    0.7452       586
weighted avg     0.7652    0.7628    0.7626       586


MODEL INFORMATION
Training samples: 2344
Test samples: 586
Number of features: 253
Initial XGBoost model trai